# Perceptrons (Minsky & Papert, 1969): Implementation / 구현

이 노트북은 Minsky & Papert의 1969년 책 *Perceptrons*의 핵심 개념을 구현합니다.
국소(local) vs 전역(global) 속성의 차이를 시각화하고, 네 도형 논증(four-figure argument)을
실험적으로 검증하며, 단층 perceptron의 근본적 한계를 보여줍니다.

This notebook implements key concepts from Minsky & Papert's 1969 book *Perceptrons*.
We visualize the local-vs-global distinction, experimentally verify the four-figure argument,
and demonstrate the fundamental limits of single-layer perceptrons.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import Tuple, List, Callable
from itertools import combinations

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Part 1: Geometric Predicates — CONVEX vs CONNECTED / 기하학적 술어

### 핵심 구분: 국소적 vs 전역적 / The Key Distinction: Local vs Global

Minsky & Papert의 가장 중요한 통찰은 기하학적 속성을 **국소적(local)**과 **전역적(global)**으로
구분한 것입니다.

- **CONVEX (볼록)**: 세 점만 검사하면 판정 가능 → order 3으로 국소적
- **CONNECTED (연결)**: 전체 구조를 봐야 판정 가능 → 어떤 차수로도 국소적이지 않음

아래에서 이 두 속성을 시각적으로 비교합니다.

Minsky & Papert's most important insight was distinguishing geometric properties as **local** vs **global**.

- **CONVEX**: Can be tested by examining only triplets of points → local of order 3
- **CONNECTED**: Requires examining the entire structure → not local of any order

Below we visually compare these two properties.

In [ ]:
def create_grid_figure(grid_size: int, active_cells: np.ndarray) -> np.ndarray:
    """Create a binary grid figure.

    Args:
        grid_size: Size of the square grid.
        active_cells: 2D boolean array marking active cells.

    Returns:
        Binary grid array.
    """
    return active_cells.astype(int)


def is_connected(grid: np.ndarray) -> bool:
    """Check if active cells in a grid form a connected component.

    Uses flood fill (BFS) from the first active cell.

    Args:
        grid: Binary 2D array.

    Returns:
        True if all active cells are connected.
    """
    if grid.sum() == 0:
        return True
    active = np.argwhere(grid == 1)
    start = tuple(active[0])
    visited = set()
    queue = [start]
    visited.add(start)
    while queue:
        r, c = queue.pop(0)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]
                    and (nr, nc) not in visited and grid[nr, nc] == 1):
                visited.add((nr, nc))
                queue.append((nr, nc))
    return len(visited) == grid.sum()


def is_convex_grid(grid: np.ndarray) -> bool:
    """Check if active cells form a convex shape (approximate for grid).

    Tests the triplet condition: for any two active cells, all cells
    on the line between them should also be active.

    Args:
        grid: Binary 2D array.

    Returns:
        True if the active region is approximately convex.
    """
    active = np.argwhere(grid == 1)
    if len(active) <= 2:
        return True
    for i in range(len(active)):
        for j in range(i + 1, len(active)):
            p, r = active[i], active[j]
            # Check points on the line segment between p and r
            n_steps = max(abs(r[0] - p[0]), abs(r[1] - p[1]))
            if n_steps == 0:
                continue
            for t in range(1, n_steps):
                q_r = int(round(p[0] + t * (r[0] - p[0]) / n_steps))
                q_c = int(round(p[1] + t * (r[1] - p[1]) / n_steps))
                if 0 <= q_r < grid.shape[0] and 0 <= q_c < grid.shape[1]:
                    if grid[q_r, q_c] == 0:
                        return False
    return True


# Create example figures
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1: Convexity examples
examples_convex = [
    ("Convex & Connected\n볼록 & 연결", np.array([
        [0,0,1,1,0,0],
        [0,1,1,1,1,0],
        [1,1,1,1,1,1],
        [0,1,1,1,1,0],
        [0,0,1,1,0,0],
    ])),
    ("Not Convex, Connected\n비볼록, 연결", np.array([
        [1,1,0,0,1,1],
        [1,1,0,0,1,1],
        [1,1,1,1,1,1],
        [1,1,0,0,1,1],
        [1,1,0,0,1,1],
    ])),
    ("Convex, Not Connected\n볼록, 비연결", np.array([
        [1,1,0,0,0,0],
        [1,1,0,0,0,0],
        [0,0,0,0,0,0],
        [0,0,0,0,1,1],
        [0,0,0,0,1,1],
    ])),
    ("Not Convex, Not Connected\n비볼록, 비연결", np.array([
        [1,0,0,0,0,1],
        [0,0,0,0,0,0],
        [0,0,0,0,0,0],
        [0,0,0,0,0,0],
        [1,0,0,0,0,1],
    ])),
]

for idx, (title, grid) in enumerate(examples_convex):
    ax = axes[0, idx]
    ax.imshow(grid, cmap='Blues', vmin=0, vmax=1, aspect='equal')
    conv = is_convex_grid(grid)
    conn = is_connected(grid)
    ax.set_title(f"{title}\nConvex={conv}, Connected={conn}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    # Add grid lines
    for i in range(grid.shape[0] + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.5)
    for j in range(grid.shape[1] + 1):
        ax.axvline(j - 0.5, color='gray', linewidth=0.5)

# Row 2: Why CONNECTED is hard for perceptrons
# The four-figure argument (simplified)
four_figs = [
    ("$X_{00}$: Disconnected\n끊김", np.array([
        [1,1,1,0,0,0,0,0,1,1,1],
    ])),
    ("$X_{10}$: Connected\n연결 (왼쪽)", np.array([
        [1,1,1,1,1,1,0,0,1,1,1],
    ])),
    ("$X_{01}$: Connected\n연결 (오른쪽)", np.array([
        [1,1,1,0,0,1,1,1,1,1,1],
    ])),
    ("$X_{11}$: Disconnected!\n끊김! (양쪽 채움)", np.array([
        [1,1,1,1,1,1,1,1,1,1,1],
    ])),
]

# Modify X11 to be disconnected (two overlapping bars)
four_figs[3] = ("$X_{11}$: Still Disconnected!\n여전히 끊김!", np.array([
    [1,1,1,1,1,0,1,1,1,1,1],
]))

for idx, (title, grid) in enumerate(four_figs):
    ax = axes[1, idx]
    # Display as colored bar
    display = np.zeros((3, grid.shape[1], 3))
    for j in range(grid.shape[1]):
        if grid[0, j] == 1:
            if j < 4:  # Left group
                display[1, j] = [0.2, 0.4, 0.8]
            elif j > 6:  # Right group
                display[1, j] = [0.8, 0.3, 0.2]
            else:  # Middle
                display[1, j] = [0.3, 0.7, 0.3]
        else:
            display[1, j] = [0.95, 0.95, 0.95]
    display[0, :] = [0.95, 0.95, 0.95]
    display[2, :] = [0.95, 0.95, 0.95]
    ax.imshow(display, aspect='auto')
    conn = is_connected(grid)
    color = 'green' if conn else 'red'
    ax.set_title(title, fontsize=10, color=color)
    ax.set_xticks([])
    ax.set_yticks([])

axes[0, 0].set_ylabel('Convexity Test\n볼록성 테스트', fontsize=11)
axes[1, 0].set_ylabel('Four-Figure Argument\n네 도형 논증', fontsize=11)

plt.suptitle('Local (CONVEX) vs Global (CONNECTED):\n'
             'Why Perceptrons Cannot Compute Connectedness',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: The Convexity Test — Order 3 Local Predicate / 볼록성 테스트

### CONVEX는 왜 국소적인가? / Why is CONVEX Local?

볼록하지 않음을 판정하려면 **세 점만** 검사하면 됩니다:
$p \in X$, $r \in X$, 그리고 $\overline{pr}$ 위의 $q \notin X$.
모든 트리플릿이 통과하면 볼록합니다.

To test non-convexity, you only need to check **triplets of points**:
$p \in X$, $r \in X$, and $q \notin X$ on segment $\overline{pr}$.
If all triplets pass, the figure is convex.

아래에서 이 트리플릿 테스트를 시각화합니다.

Below we visualize this triplet test.

In [ ]:
def visualize_convexity_test(grid: np.ndarray, ax: plt.Axes, title: str) -> None:
    """Visualize the triplet convexity test on a grid figure.

    Finds and highlights a failing triplet (p, q, r) if the figure
    is not convex, or shows a passing triplet if convex.

    Args:
        grid: Binary 2D array.
        ax: Matplotlib axes to plot on.
        title: Plot title.
    """
    ax.imshow(grid, cmap='Blues', vmin=0, vmax=1, aspect='equal')
    active = np.argwhere(grid == 1)

    # Find a failing triplet
    fail_found = False
    for i in range(len(active)):
        for j in range(i + 1, len(active)):
            p, r = active[i], active[j]
            n_steps = max(abs(r[0] - p[0]), abs(r[1] - p[1]))
            if n_steps == 0:
                continue
            for t in range(1, n_steps):
                q_r = int(round(p[0] + t * (r[0] - p[0]) / n_steps))
                q_c = int(round(p[1] + t * (r[1] - p[1]) / n_steps))
                if (0 <= q_r < grid.shape[0] and 0 <= q_c < grid.shape[1]
                        and grid[q_r, q_c] == 0):
                    # Found failing triplet
                    ax.plot([p[1], r[1]], [p[0], r[0]], 'r-', linewidth=2)
                    ax.plot(p[1], p[0], 'go', markersize=12, label='p ∈ X')
                    ax.plot(r[1], r[0], 'go', markersize=12, label='r ∈ X')
                    ax.plot(q_c, q_r, 'rx', markersize=15, markeredgewidth=3,
                            label='q ∉ X')
                    fail_found = True
                    break
            if fail_found:
                break
        if fail_found:
            break

    if not fail_found:
        # Show a passing triplet
        if len(active) >= 2:
            p, r = active[0], active[-1]
            ax.plot([p[1], r[1]], [p[0], r[0]], 'g--', linewidth=2, alpha=0.5)
            ax.plot(p[1], p[0], 'go', markersize=12)
            ax.plot(r[1], r[0], 'go', markersize=12)

    status = 'NOT CONVEX (failing triplet found)' if fail_found else 'CONVEX (all triplets pass)'
    color = 'red' if fail_found else 'green'
    ax.set_title(f"{title}\n{status}", color=color, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    for i in range(grid.shape[0] + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.5)
    for j in range(grid.shape[1] + 1):
        ax.axvline(j - 0.5, color='gray', linewidth=0.5)


# Test examples
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Convex figure (filled circle-like)
g1 = np.array([
    [0,0,1,1,1,0,0],
    [0,1,1,1,1,1,0],
    [1,1,1,1,1,1,1],
    [0,1,1,1,1,1,0],
    [0,0,1,1,1,0,0],
])
visualize_convexity_test(g1, axes[0], 'Filled Ellipse / 채워진 타원')

# Non-convex (L-shape)
g2 = np.array([
    [1,1,0,0,0,0,0],
    [1,1,0,0,0,0,0],
    [1,1,0,0,0,0,0],
    [1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1],
])
visualize_convexity_test(g2, axes[1], 'L-Shape / L자 형태')

# Non-convex (ring)
g3 = np.array([
    [0,1,1,1,1,1,0],
    [1,1,0,0,0,1,1],
    [1,0,0,0,0,0,1],
    [1,1,0,0,0,1,1],
    [0,1,1,1,1,1,0],
])
visualize_convexity_test(g3, axes[2], 'Ring / 고리')

plt.suptitle('Convexity is Local (Order 3): Only Triplets Need Checking\n'
             '볼록성은 국소적 (Order 3): 세 점 검사만으로 판정 가능',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: The Four-Figure Argument — Why CONNECTED is Not Local / 네 도형 논증

### Theorem 0.8의 핵심 증명 / The Core Proof of Theorem 0.8

지름 제한 perceptron이 연결성을 계산할 수 없다는 것을 보여주는 네 도형 논증입니다.
핵심 아이디어: 국소적 관찰자가 왼쪽/오른쪽 변화를 **독립적으로만** 감지할 수 있다면,
양쪽이 동시에 변할 때 모순이 발생합니다.

The four-figure argument proves that diameter-limited perceptrons cannot compute connectedness.
Key idea: if local observers detect left/right changes **independently**,
a contradiction arises when both change simultaneously.

$$\text{Score}(X_{11}) = \text{Score}(X_{00}) + \Delta_1 + \Delta_2 > 0$$

하지만 $X_{11}$은 끊어져 있으므로 점수가 음수여야 합니다 → **모순!**

But $X_{11}$ is disconnected, so the score should be negative → **contradiction!**

In [ ]:
def simulate_four_figure_argument(
    width: int = 30,
    diameter: int = 5,
    n_predicates: int = 200
) -> dict:
    """Simulate the four-figure argument with a diameter-limited perceptron.

    Creates four 1D figures and trains a diameter-limited perceptron,
    demonstrating the inevitable contradiction.

    Args:
        width: Width of the 1D retina.
        diameter: Maximum diameter of each partial predicate's support.
        n_predicates: Number of random partial predicates.

    Returns:
        Dict with scores and predictions for each figure.
    """
    gap_start = width // 3
    gap_end = 2 * width // 3

    # Four figures (1D binary arrays)
    x00 = np.zeros(width)
    x00[:gap_start] = 1
    x00[gap_end:] = 1  # Disconnected

    x10 = x00.copy()
    x10[gap_start:gap_start + (gap_end - gap_start) // 2] = 1  # Left bridge

    x01 = x00.copy()
    x01[gap_end - (gap_end - gap_start) // 2:gap_end] = 1  # Right bridge

    x11 = np.zeros(width)
    x11[:gap_start + (gap_end - gap_start) // 2] = 1
    x11[gap_end - (gap_end - gap_start) // 2:] = 1
    # Keep a gap in the middle to make x11 disconnected
    mid = width // 2
    x11[mid] = 0

    figures = {'X00': x00, 'X10': x10, 'X01': x01, 'X11': x11}
    labels = {'X00': 0, 'X10': 1, 'X01': 1, 'X11': 0}  # 0=disconnected, 1=connected

    # Create diameter-limited partial predicates
    phi_supports = []  # (start, end) of each predicate's support
    for _ in range(n_predicates):
        start = np.random.randint(0, width - diameter)
        phi_supports.append((start, start + diameter))

    # Compute phi values for each figure
    phi_values = {}
    for name, fig in figures.items():
        vals = []
        for start, end in phi_supports:
            # Simple predicate: proportion of active cells in support
            vals.append(fig[start:end].mean())
        phi_values[name] = np.array(vals)

    # Try to find weights that correctly classify all four
    # Use simple perceptron learning
    weights = np.random.randn(n_predicates) * 0.01
    theta = 0.0
    lr = 0.01

    for epoch in range(1000):
        for name in ['X00', 'X10', 'X01', 'X11']:
            score = np.dot(weights, phi_values[name]) - theta
            pred = 1 if score > 0 else 0
            error = labels[name] - pred
            if error != 0:
                weights += lr * error * phi_values[name]
                theta -= lr * error

    # Final scores
    results = {}
    for name in ['X00', 'X10', 'X01', 'X11']:
        score = np.dot(weights, phi_values[name]) - theta
        pred = 1 if score > 0 else 0
        correct = pred == labels[name]
        results[name] = {
            'score': score,
            'pred': pred,
            'label': labels[name],
            'correct': correct,
            'figure': figures[name]
        }

    # Classify predicates into groups
    groups = {'left': [], 'right': [], 'middle': []}
    for i, (start, end) in enumerate(phi_supports):
        center = (start + end) / 2
        if center < gap_start:
            groups['left'].append(i)
        elif center > gap_end:
            groups['right'].append(i)
        else:
            groups['middle'].append(i)

    results['groups'] = groups
    results['weights'] = weights
    results['phi_values'] = phi_values
    results['phi_supports'] = phi_supports
    return results


# Run simulation
results = simulate_four_figure_argument(width=30, diameter=5, n_predicates=200)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig_names = ['X00', 'X10', 'X01', 'X11']
fig_titles = [
    '$X_{00}$: Disconnected (should reject) / 끊김 (거부해야 함)',
    '$X_{10}$: Connected (should accept) / 연결 (수용해야 함)',
    '$X_{01}$: Connected (should accept) / 연결 (수용해야 함)',
    '$X_{11}$: Disconnected (should reject) / 끊김 (거부해야 함)',
]

for idx, (name, title) in enumerate(zip(fig_names, fig_titles)):
    ax = axes[idx // 2, idx % 2]
    r = results[name]
    x = np.arange(len(r['figure']))
    colors = ['steelblue' if v == 1 else 'lightgray' for v in r['figure']]
    ax.bar(x, np.ones(len(x)), color=colors, edgecolor='gray', linewidth=0.5)
    ax.set_ylim(0, 1.5)

    pred_str = 'ACCEPT' if r['pred'] == 1 else 'REJECT'
    correct_str = '✓' if r['correct'] else '✗ WRONG!'
    color = 'green' if r['correct'] else 'red'
    ax.set_title(f"{title}\nScore={r['score']:.3f} → {pred_str} {correct_str}",
                 color=color, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

all_correct = all(results[n]['correct'] for n in fig_names)
verdict = ("Perceptron SUCCEEDED (unlikely!)" if all_correct
           else "Perceptron FAILED — cannot classify all four correctly!")
plt.suptitle(f'Four-Figure Argument Simulation\n{verdict}',
             fontsize=14, fontweight='bold',
             color='green' if all_correct else 'red')
plt.tight_layout()
plt.show()

# Print the additivity argument
print("\n=== Four-Figure Argument Analysis ===")
print(f"Score(X00) = {results['X00']['score']:.4f} (should be < 0)")
print(f"Score(X10) = {results['X10']['score']:.4f} (should be > 0)")
print(f"Score(X01) = {results['X01']['score']:.4f} (should be > 0)")
print(f"Score(X11) = {results['X11']['score']:.4f} (should be < 0)")
delta1 = results['X10']['score'] - results['X00']['score']
delta2 = results['X01']['score'] - results['X00']['score']
print(f"\nΔ1 (left change)  = {delta1:.4f}")
print(f"Δ2 (right change) = {delta2:.4f}")
print(f"Score(X00) + Δ1 + Δ2 = {results['X00']['score'] + delta1 + delta2:.4f}")
print(f"Actual Score(X11)     = {results['X11']['score']:.4f}")
print(f"\nThe perceptron cannot make Score(X11) < 0")
print(f"when Score(X10) > 0 and Score(X01) > 0!")

## Part 4: Order-Limited Perceptrons and XOR / 차수 제한 Perceptron과 XOR

### XOR: 가장 단순한 선형 분리 불가능 문제 / The Simplest Linearly Inseparable Problem

XOR은 order-1 perceptron(각 부분 술어가 하나의 점만 보는 perceptron)으로는
절대 풀 수 없습니다. 이것은 Minsky & Papert이 지적한 한계의 가장 단순한 예입니다.

XOR cannot be solved by an order-1 perceptron (where each partial predicate
examines only one point). This is the simplest example of the limits Minsky & Papert identified.

$$\text{XOR}(x_1, x_2) = 1 \iff a_1 x_1 + a_2 x_2 > \theta \quad \text{has NO solution}$$

하지만 2층 perceptron(Gamba perceptron)으로는 해결 가능합니다.

However, a two-layer perceptron (Gamba perceptron) CAN solve it.

In [ ]:
# XOR: exhaustive proof that no single-layer solution exists
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

def check_linear_separability(
    X: np.ndarray, y: np.ndarray, n_tries: int = 100000
) -> Tuple[bool, np.ndarray, float]:
    """Check if a dataset is linearly separable by random search.

    Args:
        X: Input data of shape (n_samples, n_features).
        y: Binary labels of shape (n_samples,).
        n_tries: Number of random weight vectors to try.

    Returns:
        Tuple of (is_separable, best_weights, best_threshold).
    """
    best_acc = 0
    best_w, best_t = None, None
    for _ in range(n_tries):
        w = np.random.randn(X.shape[1])
        t = np.random.randn()
        preds = (X @ w > t).astype(int)
        acc = np.mean(preds == y)
        if acc > best_acc:
            best_acc = acc
            best_w, best_t = w.copy(), t
        if acc == 1.0:
            return True, best_w, best_t
    return False, best_w, best_t


# Test XOR, AND, OR
problems = {
    'AND': np.array([0, 0, 0, 1]),
    'OR': np.array([0, 1, 1, 1]),
    'XOR': np.array([0, 1, 1, 0]),
    'XNOR': np.array([1, 0, 0, 1]),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for idx, (name, y) in enumerate(problems.items()):
    sep, w, t = check_linear_separability(X_xor, y)
    ax = axes[idx]

    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300),
                         np.linspace(-0.5, 1.5, 300))
    if w is not None:
        Z = (np.c_[xx.ravel(), yy.ravel()] @ w > t).reshape(xx.shape).astype(float)
        ax.contourf(xx, yy, Z, alpha=0.2, levels=[-0.5, 0.5, 1.5],
                    colors=['#ffcccc', '#ccffcc'])
        if abs(w[1]) > 1e-10:
            x_line = np.linspace(-0.5, 1.5, 100)
            y_line = (t - w[0] * x_line) / w[1]
            mask = (y_line > -0.5) & (y_line < 1.5)
            ax.plot(x_line[mask], y_line[mask], 'k-', linewidth=2)

    for i in range(4):
        color = 'green' if y[i] == 1 else 'red'
        marker = 'o' if y[i] == 1 else 's'
        ax.scatter(X_xor[i, 0], X_xor[i, 1], c=color, s=200,
                   marker=marker, edgecolors='black', linewidth=2, zorder=5)

    status = '✓ Separable' if sep else '✗ NOT Separable'
    color = 'green' if sep else 'red'
    ax.set_title(f'{name}\n{status}', color=color, fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Linear Separability: AND and OR are separable, XOR and XNOR are NOT\n'
             '선형 분리 가능성: AND, OR은 가능, XOR, XNOR은 불가능',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: Perceptron Repertoire $L(\Phi)$ — What Can Be Computed? / 레퍼토리

### perceptron이 계산할 수 있는 함수의 비율 / Fraction of Computable Functions

Minsky & Papert의 핵심 주장: 물리적으로 실현 가능한 perceptron의 레퍼토리 $L(\Phi)$는
가능한 모든 술어의 극히 작은 부분집합입니다.

아래에서 $n$개의 입력에 대해 가능한 $2^{2^n}$개의 Boolean 함수 중
선형 분리 가능한 함수의 비율을 계산합니다.

Minsky & Papert's key claim: the repertoire $L(\Phi)$ of any physically realizable
perceptron is a tiny subset of all possible predicates.

Below we compute the fraction of all $2^{2^n}$ Boolean functions on $n$ inputs
that are linearly separable.

In [ ]:
def count_linearly_separable(n: int) -> Tuple[int, int]:
    """Count linearly separable Boolean functions on n binary inputs.

    Uses exhaustive enumeration for small n.

    Args:
        n: Number of input bits.

    Returns:
        Tuple of (n_separable, n_total).
    """
    X = np.array(list(np.ndindex(*([2] * n))))
    n_total = 2 ** len(X)
    n_sep = 0

    for func_idx in range(n_total):
        y = np.array([(func_idx >> i) & 1 for i in range(len(X))])
        # Try random search for linear separability
        found = False
        for _ in range(5000):
            w = np.random.randn(n)
            t = np.random.randn()
            preds = (X @ w > t).astype(int)
            if np.all(preds == y):
                found = True
                break
        if found:
            n_sep += 1

    return n_sep, n_total


# Compute for n=1,2,3,4
# Known exact values: n=1: 4/4, n=2: 14/16, n=3: 104/256, n=4: 1882/65536
known_values = {
    1: (4, 4),
    2: (14, 16),
    3: (104, 256),
    4: (1882, 65536),
    5: (94572, 2**32),
    6: (15028134, 2**64),
}

ns = [1, 2, 3, 4, 5, 6]
fractions = [known_values[n][0] / known_values[n][1] for n in ns]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: fraction plot
ax1.plot(ns, fractions, 'ro-', markersize=10, linewidth=2)
ax1.set_xlabel('Number of inputs ($n$)')
ax1.set_ylabel('Fraction linearly separable')
ax1.set_title('Fraction of Boolean Functions that are\n'
              'Linearly Separable (= Perceptron-Computable)\n'
              '선형 분리 가능한 Boolean 함수의 비율')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
for n, f in zip(ns, fractions):
    ax1.annotate(f'{f:.2e}', (n, f), textcoords='offset points',
                 xytext=(10, 5), fontsize=9)

# Right: counts
data = []
for n in ns[:4]:
    sep, total = known_values[n]
    data.append([n, total, sep, total - sep])

x_pos = np.arange(4)
totals = [d[1] for d in data]
seps = [d[2] for d in data]
ax2.bar(x_pos - 0.15, totals, 0.3, label='Total Boolean functions / 전체',
        color='lightcoral', edgecolor='black')
ax2.bar(x_pos + 0.15, seps, 0.3, label='Linearly separable / 선형 분리 가능',
        color='lightgreen', edgecolor='black')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'n={d[0]}' for d in data])
ax2.set_ylabel('Count (log scale)')
ax2.set_yscale('log')
ax2.set_title('Total vs Linearly Separable Boolean Functions\n'
              '전체 vs 선형 분리 가능 Boolean 함수')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Linearly separable Boolean functions:")
print(f"{'n':>3} | {'Total':>12} | {'Separable':>12} | {'Fraction':>12}")
print("-" * 50)
for n in ns:
    sep, total = known_values[n]
    print(f"{n:>3} | {total:>12} | {sep:>12} | {sep/total:>12.2e}")
print("\nAs n grows, the fraction drops SUPER-exponentially!")
print("This is Minsky & Papert's point: L(Φ) is a tiny fraction of all predicates.")

## Part 6: Multi-Layer Solution — Breaking the Minsky-Papert Barrier / 다층 해결책

### 다층 Perceptron으로 한계 극복 / Overcoming the Limits with Multi-Layer Perceptrons

Minsky & Papert의 증명은 **Stage I이 독립적이고 Stage II가 선형**인 경우에만 적용됩니다.
다층 구조(hidden layers)를 추가하면 이 제약이 깨집니다.

아래에서 XOR과 연결성(connectedness) 문제를 다층 네트워크로 해결합니다.

Minsky & Papert's proofs apply only when **Stage I is independent and Stage II is linear**.
Adding hidden layers breaks these constraints.

Below we solve XOR and connectedness with multi-layer networks.

In [ ]:
class TwoLayerPerceptron:
    """A simple two-layer perceptron (MLP) for solving XOR.

    Architecture: input(2) -> hidden(2) -> output(1)
    Uses sigmoid activation and gradient descent.
    """

    def __init__(self, n_hidden: int = 2, lr: float = 1.0):
        self.W1 = np.random.randn(2, n_hidden) * 0.5
        self.b1 = np.zeros(n_hidden)
        self.W2 = np.random.randn(n_hidden, 1) * 0.5
        self.b2 = np.zeros(1)
        self.lr = lr

    def sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Sigmoid activation."""
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Forward pass."""
        h = self.sigmoid(X @ self.W1 + self.b1)
        o = self.sigmoid(h @ self.W2 + self.b2)
        return h, o

    def train(self, X: np.ndarray, y: np.ndarray, n_epochs: int = 5000) -> List[float]:
        """Train with backpropagation."""
        losses = []
        y = y.reshape(-1, 1)
        for _ in range(n_epochs):
            h, o = self.forward(X)
            loss = np.mean((o - y) ** 2)
            losses.append(loss)

            # Backprop
            d_o = (o - y) * o * (1 - o)
            d_W2 = h.T @ d_o
            d_b2 = d_o.sum(axis=0)
            d_h = d_o @ self.W2.T * h * (1 - h)
            d_W1 = X.T @ d_h
            d_b1 = d_h.sum(axis=0)

            self.W2 -= self.lr * d_W2
            self.b2 -= self.lr * d_b2
            self.W1 -= self.lr * d_W1
            self.b1 -= self.lr * d_b1
        return losses

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Binary prediction."""
        _, o = self.forward(X)
        return (o > 0.5).astype(int).flatten()


# Train on XOR
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

mlp = TwoLayerPerceptron(n_hidden=4, lr=2.0)
losses = mlp.train(X_xor, y_xor, n_epochs=5000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Loss curve
ax1.plot(losses, 'b-', linewidth=1)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('XOR Learning Curve (Two-Layer Perceptron)\nXOR 학습 곡선 (2층 Perceptron)')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Right: Decision boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid_input = np.c_[xx.ravel(), yy.ravel()]
_, Z = mlp.forward(grid_input)
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, levels=20, cmap='RdYlGn', alpha=0.7)
ax2.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

for i in range(4):
    color = 'green' if y_xor[i] == 1 else 'red'
    marker = 'o' if y_xor[i] == 1 else 's'
    ax2.scatter(X_xor[i, 0], X_xor[i, 1], c=color, s=200,
               marker=marker, edgecolors='black', linewidth=2, zorder=5)

preds = mlp.predict(X_xor)
acc = np.mean(preds == y_xor.astype(int))
ax2.set_title(f'XOR Decision Boundary (Accuracy: {acc:.0%})\n'
              f'XOR 결정 경계 — 비선형 경계로 해결!',
              color='green' if acc == 1.0 else 'red')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_xlim(-0.5, 1.5)
ax2.set_ylim(-0.5, 1.5)

plt.suptitle('Breaking the Minsky-Papert Barrier: Multi-Layer Perceptron Solves XOR\n'
             'Minsky-Papert 장벽 돌파: 다층 Perceptron이 XOR을 해결',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nPredictions: {preds}")
print(f"Labels:      {y_xor.astype(int)}")
print(f"Accuracy:    {acc:.0%}")

## Part 7: sklearn Comparison — From Perceptron to MLP / 현대 구현 비교

### Minsky-Papert의 한계 vs 현대 해결책 / The Limit vs Modern Solution

scikit-learn의 `Perceptron`(단층)과 `MLPClassifier`(다층)를 비교하여
Minsky-Papert이 지적한 한계가 다층 구조로 어떻게 극복되는지 확인합니다.

We compare sklearn's `Perceptron` (single-layer) with `MLPClassifier` (multi-layer)
to verify that multi-layer architectures overcome the Minsky-Papert limitation.

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons, make_circles

# Generate non-linearly separable datasets
datasets = {
    'XOR': (X_xor, y_xor.astype(int)),
    'Moons': make_moons(n_samples=200, noise=0.15, random_state=42),
    'Circles': make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for col, (name, (X, y)) in enumerate(datasets.items()):
    for row, (model_name, model) in enumerate([
        ('Perceptron (1-layer)\nSingle-layer / 단층', Perceptron(max_iter=1000)),
        ('MLP (2-layer)\nMulti-layer / 다층', MLPClassifier(
            hidden_layer_sizes=(10,), max_iter=2000, random_state=42)),
    ]):
        ax = axes[row, col]
        model.fit(X, y)
        acc = model.score(X, y)

        xx, yy = np.meshgrid(
            np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
            np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        ax.contourf(xx, yy, Z, alpha=0.3, levels=[-0.5, 0.5, 1.5],
                    colors=['#ffcccc', '#ccffcc'])
        ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)
        scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn',
                            s=30, edgecolors='black', linewidth=0.5)

        color = 'green' if acc > 0.95 else 'red'
        ax.set_title(f'{model_name}\n{name}: {acc:.0%}', color=color, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Single-Layer (Minsky-Papert limit) vs Multi-Layer (limit broken)\n'
             '단층 (Minsky-Papert 한계) vs 다층 (한계 극복)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary: Minsky & Papert's Legacy / 요약: Minsky & Papert의 유산

| Concept / 개념 | Minsky & Papert (1969) | Modern Equivalent / 현대 동등 | Resolution / 해결 |
|---|---|---|---|
| Local vs Global | CONVEX (local, order 3) vs CONNECTED (global) | Receptive field size in CNNs | Stacking layers increases effective receptive field / 층을 쌓으면 실효 수용야 증가 |
| Linear inseparability | XOR cannot be solved by order-1 perceptron | Kernel methods, deep networks | Hidden layers create nonlinear decision boundaries / 은닉층이 비선형 결정 경계 생성 |
| Limited repertoire $L(\Phi)$ | Fraction of computable functions drops super-exponentially | Universal approximation theorem | Multi-layer networks are universal approximators / 다층 네트워크는 범용 근사기 |
| Convergence time | Can be worse than exhaustive search | Modern optimizers (Adam, etc.) | **#6 Rumelhart et al. (1986)** — backpropagation |
| "Significant learning needs structure" | Random $\Phi$ cannot solve high-order problems | Inductive bias, architecture design | CNNs, Transformers — task-appropriate architectures / 과제에 맞는 아키텍처 설계 |

### 다음 논문 / Next Paper

**#5 Hopfield — "Neural Networks and Physical Systems with Emergent Collective Computational Abilities" (1982)**

AI 겨울을 끝내고 신경망 부활의 시작을 알린 논문입니다.
물리학의 에너지 최소화 원리를 신경망에 적용하여, Minsky-Papert의 비판을
정면으로 우회하는 새로운 패러다임을 제시합니다.

The paper that ended the AI winter and began the neural network renaissance.
It applies physics energy minimization to neural networks,
sidestepping Minsky-Papert's critique with a new paradigm.